# 05 · Multi-agent debate (D0 control, majority vote, judge)

Three agents answer independently (round 0, using the prompt set by `debate.round0_setting`; default: the same few-shot prompt and example set as single-agent S2 for that seed), then revise for up to two rounds while seeing the anonymised answers of the others; the debate stops early on unanimity. From one run we derive **D0** (round-0 majority vote, no revision), **debate + vote**, and **debate + judge** (judge from a family that does not debate). Compositions: homogeneous (3 × one family) and heterogeneous (one agent per family). Resumable.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
from mad_indo.llm import setup_ollama, OllamaClient
lc, dc = cfg["llm"], cfg["debate"]
setup_ollama(list(lc["models"].values()) + [lc["judge_model"]], host=lc["host"],
             num_parallel=lc["parallel_requests"], max_loaded_models=4)
client = OllamaClient(lc["host"], num_ctx=lc["num_ctx"], num_predict=lc["num_predict"])

Note: server already running; restart the Colab session if OLLAMA_NUM_PARALLEL must change.
Pulling qwen2.5:7b ...
Pulling llama3.1:8b ...
Pulling gemma2:9b ...
Pulling mistral:7b ...
Ollama ready | ollama version is 0.40.16 | models: qwen2.5:7b, llama3.1:8b, gemma2:9b, mistral:7b


In [3]:
RUN_TASKS = ["smsa", "emot"]
compositions = {f"homo-{k}": [(f"{k}{i}", lc["models"][k]) for i in range(dc["n_agents"])]
                for k in dc["compositions"]["homogeneous"]}
compositions["hetero"] = [(k, lc["models"][k]) for k in dc["compositions"]["heterogeneous"]]
RUN_COMPOSITIONS = list(compositions)            # e.g. ["hetero"] for one session
compositions

{'homo-qwen': [('qwen0', 'qwen2.5:7b'),
  ('qwen1', 'qwen2.5:7b'),
  ('qwen2', 'qwen2.5:7b')],
 'homo-llama': [('llama0', 'llama3.1:8b'),
  ('llama1', 'llama3.1:8b'),
  ('llama2', 'llama3.1:8b')],
 'homo-gemma': [('gemma0', 'gemma2:9b'),
  ('gemma1', 'gemma2:9b'),
  ('gemma2', 'gemma2:9b')],
 'hetero': [('qwen', 'qwen2.5:7b'),
  ('llama', 'llama3.1:8b'),
  ('gemma', 'gemma2:9b')]}

In [4]:
from mad_indo.runners import run_debate
out_dir = RESULTS / "debate"; out_dir.mkdir(exist_ok=True)
for task in RUN_TASKS:
    L = cfg["tasks"][task]["labels"]; test = data.load_split(task, "test", limit=LIMIT)
    train = data.load_split(task, "train")
    for comp in RUN_COMPOSITIONS:
        seeds = dc["seeds"] + (dc["extra_seeds"] if comp in dc["extra_seed_compositions"] else [])
        for seed in seeds:
            print(f"{task} | {comp} | seed {seed}")
            run_debate(client, comp, compositions[comp], task, L, test, seed, cfg,
                       out_dir / f"{task}_{comp}.jsonl", judge_model=lc["judge_model"],
                       workers=lc["parallel_requests"], train_df=train)

smsa | homo-qwen | seed 11
smsa | homo-llama | seed 11
smsa | homo-llama | seed 22
smsa | homo-llama | seed 33
smsa | homo-gemma | seed 11
smsa | hetero | seed 11
smsa | hetero | seed 22
smsa | hetero | seed 33
emot | homo-qwen | seed 11
emot | homo-llama | seed 11
emot | homo-llama | seed 22
emot | homo-llama | seed 33
emot | homo-gemma | seed 11
emot | hetero | seed 11
emot | hetero | seed 22
emot | hetero | seed 33
